In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

file_path = "abfss://mariiasirska@dlsua5816bd.dfs.core.windows.net/1. Dataset.csv"
df_raw = spark.read.csv(file_path, header=True, inferSchema=True)

df_bronze = df_raw.withColumn("source_filename", col("_metadata.file_path")) \
                  .withColumn("ingestion_timestamp", current_timestamp()) \
                  .withColumn("load_date", current_date())

df_bronze.createOrReplaceTempView("updates")

table_name = "dbr_dev_ua5816bd.mariiasirska_bronze.badminton_players"

if not spark.catalog.tableExists(table_name):
    df_bronze.write.format("delta").saveAsTable(table_name)
else:
    spark.sql(f"""
        MERGE INTO {table_name} target
        USING updates
        ON target.Name = updates.Name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)